# Final Live Demonstration: External APIs + Dataset Simulation

This notebook is the runnable presentation layer for the final demo. It shows three live-facing paths from inside Jupyter:

1. **External live operational data** from AviationWeather.gov METAR and AeroDataBox airport flights.
2. **Dataset simulation stream** from real Gold feature rows, published to Kafka and scored by the API.
3. **10x API load test** plus Prometheus/Grafana health checks.

The production code remains in `.py` modules, but every demo action below is executable as a notebook cell.

In [1]:
from pathlib import Path
import json
import os
import subprocess
import time

PROJECT_ROOT = Path('/workspace') if Path('/workspace').exists() else Path.cwd().resolve().parents[1]
os.chdir(PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)

LIVE_DIR = PROJECT_ROOT / 'data/local_cache/live_predictions'
LIVE_DIR.mkdir(parents=True, exist_ok=True)

API_URL = 'http://aviation-api:3000/predict'
PROM_URL = 'http://prometheus:9090'
GRAFANA_URL = 'http://grafana:3000'

Project root: /workspace


## Helper Functions

These helpers keep notebook cells readable while still showing the exact command output. They also load `.env` values for AeroDataBox without printing secrets.

In [2]:
def run(command, timeout=300):
    print('$', command)
    completed = subprocess.run(
        command,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=timeout,
    )
    print(completed.stdout)
    if completed.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {completed.returncode}')
    return completed.stdout


def load_dotenv(path='.env'):
    env_path = PROJECT_ROOT / path
    if not env_path.exists():
        print('.env not found; continuing with current environment')
        return
    for raw_line in env_path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        os.environ[key] = value.strip().strip('"').strip("'")
    print('Loaded .env keys. AeroDataBox key configured:', bool(os.environ.get('AERODATABOX_API_KEY')))


def latest_jsonl(path, n=3):
    path = Path(path)
    if not path.exists():
        print('Missing file:', path)
        return []
    lines = path.read_text().splitlines()[-n:]
    events = [json.loads(line) for line in lines]
    for event in events:
        print(json.dumps(event, indent=2)[:4000])
    return events

load_dotenv()

Loaded .env keys. AeroDataBox key configured: True


## 1. Service Health

The demo needs the API, Kafka, Prometheus, Grafana, Jupyter, Spark, and MinIO.

In [3]:
import socket
import requests

checks = {
    'aviation-api': ('aviation-api', 3000, 'http://aviation-api:3000/metrics'),
    'kafka': ('kafka', 9092, None),
    'prometheus': ('prometheus', 9090, 'http://prometheus:9090/-/healthy'),
    'grafana': ('grafana', 3000, 'http://grafana:3000/api/health'),
    'minio': ('minio', 9000, 'http://minio:9000/minio/health/live'),
    'spark-master': ('spark-master', 7077, None),
}

for name, (host, port, url) in checks.items():
    with socket.create_connection((host, port), timeout=5):
        socket_status = 'tcp-ok'
    http_status = 'not-checked'
    if url:
        response = requests.get(url, timeout=10)
        http_status = f'http-{response.status_code}'
        response.raise_for_status()
    print(f'{name}: {socket_status}, {http_status}')

aviation-api: tcp-ok, http-200
kafka: tcp-ok, not-checked
prometheus: tcp-ok, http-200
grafana: tcp-ok, http-200
minio: tcp-ok, http-200
spark-master: tcp-ok, not-checked


## 2. External Live Operational Prediction

This cell fetches current aviation weather from **AviationWeather.gov** and live airport arrivals/departures from **AeroDataBox**, then calls the deployed API.

The `--require-flight-api` flag intentionally fails if the AeroDataBox key is missing, so this cell proves the live flight provider is actually active.

In [4]:
external_path = LIVE_DIR / 'notebook_external_operational.jsonl'
cmd = (
    'set -a && . ./.env && set +a && '
    'python -m api.live_operational_predict '
    '--airport JFK '
    '--require-flight-api '
    f'--output-jsonl {external_path} '
    f'--api-url {API_URL}'
)
run(cmd, timeout=180)

$ set -a && . ./.env && set +a && python -m api.live_operational_predict --airport JFK --require-flight-api --output-jsonl /workspace/data/local_cache/live_predictions/notebook_external_operational.jsonl --api-url http://aviation-api:3000/predict


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/05 13:22:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/05 13:22:06 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).
26/06/05 13:22:16 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties

[Stage 0:>                                                          (0 + 1) / 1]

                                                                                

[Stage 1:>                                                          (0 + 1) / 1]

                                                                 

'WARNING: Using incubator modules: jdk.incubator.vector\nUsing Spark\'s default log4j profile: org/apache/spark/log4j2-defaults.properties\nSetting default log level to "WARN".\nTo adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).\n26/06/05 13:22:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable\n26/06/05 13:22:06 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).\n26/06/05 13:22:16 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties\n\n[Stage 0:>                                                          (0 + 1) / 1]\n\n                                                                                \n\n[Stage 1:>                                                          (0 + 1) / 

### External Live Evidence

The JSONL event includes the METAR, AeroDataBox operations summary, model features, and prediction response.

In [5]:
events = latest_jsonl(external_path, n=1)
if events:
    event = events[-1]
    print('AeroDataBox available:', event['aerodatabox_operations']['available'])
    print('Total live flights:', event['aerodatabox_operations'].get('total_flights'))
    print('METAR:', event['aviation_weather_metar']['raw'])
    print('Prediction:', event['api_response'])

{
  "aerodatabox_operations": {
    "arrivals_count": 146,
    "available": true,
    "average_observed_delay_minutes": null,
    "cancelled_count": 4,
    "delayed_15min_count": 0,
    "departures_count": 221,
    "diverted_count": 0,
    "provider": "AeroDataBox",
    "sample_flights": [
      {
        "actual_local": null,
        "airport": null,
        "number": "DL 668",
        "revised_local": null,
        "scheduled_local": null,
        "status": "Departed"
      },
      {
        "actual_local": null,
        "airport": null,
        "number": "DL 2049",
        "revised_local": null,
        "scheduled_local": null,
        "status": "Departed"
      },
      {
        "actual_local": null,
        "airport": null,
        "number": "B6 411",
        "revised_local": null,
        "scheduled_local": null,
        "status": "Departed"
      },
      {
        "actual_local": null,
        "airport": null,
        "number": "AS 31",
        "revised_local": null,
        

## 3. Dataset Simulation Stream

This cell replays real Gold feature rows from the one-year lakehouse as if they were arriving live. Each replay event is:

1. Published to Kafka topic `simulation.prediction.requests`.
2. Sent to the prediction API with source label `dataset_simulation`.
3. Saved to JSONL for audit evidence.

This proves that if live data arrives in the same feature contract as the project dataset, the deployed system can continuously score it.

In [6]:
simulation_path = LIVE_DIR / 'notebook_gold_simulation.jsonl'
cmd = (
    'python -m api.simulate_gold_stream_predict '
    '--year 2024 '
    '--month 1 '
    '--limit 25 '
    '--delay-seconds 0.1 '
    f'--output-jsonl {simulation_path} '
    f'--api-url {API_URL}'
)
run(cmd, timeout=300)

$ python -m api.simulate_gold_stream_predict --year 2024 --month 1 --limit 25 --delay-seconds 0.1 --output-jsonl /workspace/data/local_cache/live_predictions/notebook_gold_simulation.jsonl --api-url http://aviation-api:3000/predict


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/05 13:22:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/05 13:22:34 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).
26/06/05 13:22:44 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties

[Stage 0:>                                                          (0 + 1) / 1]

                                                                                

[Stage 1:>                                                          (0 + 2) / 2]

                                                                 

'WARNING: Using incubator modules: jdk.incubator.vector\nUsing Spark\'s default log4j profile: org/apache/spark/log4j2-defaults.properties\nSetting default log level to "WARN".\nTo adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).\n26/06/05 13:22:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable\n26/06/05 13:22:34 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).\n26/06/05 13:22:44 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties\n\n[Stage 0:>                                                          (0 + 1) / 1]\n\n                                                                                \n\n[Stage 1:>                                                          (0 + 2) / 

### Simulation Evidence

The output below summarizes the latest replayed Gold rows and the model predictions.

In [7]:
events = latest_jsonl(simulation_path, n=5)
summary = []
for event in events:
    source = event['source_event']
    response = event['api_response']
    summary.append({
        'sequence': event['sequence'],
        'flight_date': source['flight_date'],
        'origin': source['origin'],
        'destination': source['destination'],
        'actual_label': source['label'],
        'prediction': response['prediction'],
        'risk_band': response['risk_band'],
        'probability': response['disruption_probability'],
    })
summary

{
  "api_response": {
    "disruption_probability": 0.457649,
    "model_name": "aviation-disruption-balanced-logistic",
    "prediction": 0,
    "risk_band": "low",
    "source": "dataset_simulation"
  },
  "sequence": 21,
  "source": "dataset_simulation",
  "source_event": {
    "dataset_month": 1,
    "dataset_year": 2024,
    "destination": "BUR",
    "event_time_utc": "2026-06-05T13:22:57.582898+00:00",
    "features": {
      "cape_j_kg_max": 0.00048828125,
      "day_of_month": 1.0,
      "day_of_week": 2.0,
      "distance_miles": 672.0,
      "precipitation_mm_sum": 0.0,
      "scheduled_departure_hour_local": 14.0,
      "surface_pressure_pa_avg": 81658.06217447917,
      "temperature_c_avg": 0.7658846537272362,
      "total_cloud_cover_avg": 0.44466401388247806,
      "wind_gust_kts_max": 10.210864112701415,
      "wind_speed_kts_max": 4.423471153902834
    },
    "flight_date": "2024-01-01",
    "label": 0.0,
    "origin": "ABQ",
    "stream_type": "gold_feature_replay"
  }

[{'sequence': 21,
  'flight_date': '2024-01-01',
  'origin': 'ABQ',
  'destination': 'BUR',
  'actual_label': 0.0,
  'prediction': 0,
  'risk_band': 'low',
  'probability': 0.457649},
 {'sequence': 22,
  'flight_date': '2024-01-01',
  'origin': 'ABQ',
  'destination': 'BWI',
  'actual_label': 1.0,
  'prediction': 1,
  'risk_band': 'high',
  'probability': 0.503831},
 {'sequence': 23,
  'flight_date': '2024-01-01',
  'origin': 'ABQ',
  'destination': 'DAL',
  'actual_label': 0.0,
  'prediction': 0,
  'risk_band': 'low',
  'probability': 0.455362},
 {'sequence': 24,
  'flight_date': '2024-01-01',
  'origin': 'ABQ',
  'destination': 'DAL',
  'actual_label': 0.0,
  'prediction': 1,
  'risk_band': 'high',
  'probability': 0.5403},
 {'sequence': 25,
  'flight_date': '2024-01-01',
  'origin': 'ABQ',
  'destination': 'DAL',
  'actual_label': 0.0,
  'prediction': 0,
  'risk_band': 'low',
  'probability': 0.372944}]

## 4. Kafka Topic Check

This verifies that simulation events are visible in Kafka, not only in API logs.

In [8]:
from kafka import KafkaConsumer
from kafka.admin import KafkaAdminClient

admin = KafkaAdminClient(bootstrap_servers='kafka:9092', client_id='notebook-topic-check')
topics = sorted(admin.list_topics())
admin.close()
print('Simulation topic exists:', 'simulation.prediction.requests' in topics)
print([topic for topic in topics if 'simulation' in topic])

consumer = KafkaConsumer(
    'simulation.prediction.requests',
    bootstrap_servers='kafka:9092',
    auto_offset_reset='earliest',
    enable_auto_commit=False,
    consumer_timeout_ms=5000,
    value_deserializer=lambda value: json.loads(value.decode('utf-8')),
)
messages = []
for message in consumer:
    messages.append(message.value)
    if len(messages) >= 2:
        break
consumer.close()
for message in messages:
    print(json.dumps(message, indent=2)[:3000])

Simulation topic exists: True
['simulation.prediction.requests']
{
  "api_response": {
    "disruption_probability": 0.410752,
    "model_name": "aviation-disruption-balanced-logistic",
    "prediction": 0,
    "risk_band": "low",
    "source": "dataset_simulation"
  },
  "sequence": 1,
  "source": "dataset_simulation",
  "source_event": {
    "dataset_month": 1,
    "dataset_year": 2024,
    "destination": "ATL",
    "event_time_utc": "2026-06-05T13:03:41.090516+00:00",
    "features": {
      "cape_j_kg_max": 19.43798828125,
      "day_of_month": 1.0,
      "day_of_week": 2.0,
      "distance_miles": 692.0,
      "precipitation_mm_sum": 0.8359700441360474,
      "scheduled_departure_hour_local": 12.0,
      "surface_pressure_pa_avg": 98923.81868489583,
      "temperature_c_avg": 3.5790191650390852,
      "total_cloud_cover_avg": 0.7983369752764702,
      "wind_gust_kts_max": 11.410444389801025,
      "wind_speed_kts_max": 4.5227931991766015
    },
    "flight_date": "2024-01-01",
   

## 5. 10x API Load Test

This is the required stability demonstration: 100 prediction requests at concurrency 10.

In [9]:
run('python3 api/load_test.py --url http://aviation-api:3000/predict --requests 100 --concurrency 10', timeout=120)

$ python3 api/load_test.py --url http://aviation-api:3000/predict --requests 100 --concurrency 10


Successful requests: 100/100
Concurrency: 10x
Mean latency ms: 58.98
Max latency ms: 179.73



'Successful requests: 100/100\nConcurrency: 10x\nMean latency ms: 58.98\nMax latency ms: 179.73\n'

## 6. Prometheus Metrics

These queries prove that Prometheus sees separate sources for dataset simulation, external live operational data, and direct API/load-test traffic.

In [10]:
import requests

def prom_query(query):
    response = requests.get(f'{PROM_URL}/api/v1/query', params={'query': query}, timeout=20)
    response.raise_for_status()
    data = response.json()['data']['result']
    print(query)
    print(json.dumps(data, indent=2))
    return data

prom_query('sum by (source, risk_band) (aviation_prediction_requests_by_source_total)')
prom_query('sum(bentoml_service_request_total{http_response_code!="200"})')
prom_query('histogram_quantile(0.95, sum(rate(aviation_prediction_latency_seconds_bucket[1m])) by (le))')

sum by (source, risk_band) (aviation_prediction_requests_by_source_total)
[
  {
    "metric": {
      "risk_band": "low",
      "source": "dataset_simulation"
    },
    "value": [
      1780665780.02,
      "149"
    ]
  },
  {
    "metric": {
      "risk_band": "high",
      "source": "dataset_simulation"
    },
    "value": [
      1780665780.02,
      "36"
    ]
  },
  {
    "metric": {
      "risk_band": "high",
      "source": "direct_api"
    },
    "value": [
      1780665780.02,
      "175"
    ]
  },
  {
    "metric": {
      "risk_band": "low",
      "source": "external_live_operational"
    },
    "value": [
      1780665780.02,
      "3"
    ]
  }
]
sum(bentoml_service_request_total{http_response_code!="200"})
[
  {
    "metric": {},
    "value": [
      1780665780.026,
      "0"
    ]
  }
]
histogram_quantile(0.95, sum(rate(aviation_prediction_latency_seconds_bucket[1m])) by (le))
[
  {
    "metric": {},
    "value": [
      1780665780.032,
      "0.00475"
    ]
  }
]


[{'metric': {}, 'value': [1780665780.032, '0.00475']}]

## 7. Grafana Dashboards

Open Grafana and show both dashboards:

- `Aviation Disruption Live API`
- `Aviation Dataset Simulation Stream`

Browser URL: http://localhost:3001

In [11]:
response = requests.get(f'{GRAFANA_URL}/api/health', timeout=20)
print(response.text)
response.raise_for_status()
print('Grafana dashboards to show:')
print('- Aviation Disruption Live API')
print('- Aviation Dataset Simulation Stream')

{
  "database": "ok",
  "version": "12.1.0",
  "commit": "ccd7b6ce7ea6184b8c7eb1de044174147dd9a648"
}
Grafana dashboards to show:
- Aviation Disruption Live API
- Aviation Dataset Simulation Stream


## 8. Final Talking Points

- Historical training and evaluation use real ARCO-ERA5 weather and BTS flight outcomes.
- External live demo uses AviationWeather.gov + AeroDataBox and scores current JFK operational context.
- Dataset simulation demo replays real Gold feature rows through Kafka and the deployed API.
- Grafana separates prediction sources and shows request rate, latency, failures, risk split, and latest probability.
- The 10x load test demonstrates API stability under concurrent requests.